# Face Recognition System Implementation (PyTorch)

This notebook implements the face recognition system using `facenet-pytorch`.

**Steps:**
1.  Setup Environment & Imports
2.  Define Configuration (Device, Paths)
3.  Load Dataset & Detect Faces (MTCNN)
4.  Generate Embeddings (InceptionResnetV1)
5.  Train Classifier (SVM)
6.  Save Model

In [ ]:
# Install dependencies if not already installed
#!pip install facenet-pytorch opencv-python scikit-learn matplotlib

     ---------------------------------------- 0.0/1.9 MB ? eta -:--:--
     ---------- ----------------------------- 0.5/1.9 MB 16.2 MB/s eta 0:00:01
     --------------------- ------------------ 1.0/1.9 MB 16.1 MB/s eta 0:00:01
     --------------------------- ------------ 1.3/1.9 MB 13.6 MB/s eta 0:00:01
     -------------------------------- ------- 1.5/1.9 MB 12.3 MB/s eta 0:00:01
     -------------------------------------- - 1.8/1.9 MB 10.5 MB/s eta 0:00:01
     ---------------------------------------- 1.9/1.9 MB 8.5 MB/s eta 0:00:00
     ---------------------------------------- 0.0/198.6 MB ? eta -:--:--
     ---------------------------------------- 0.2/198.6 MB 7.3 MB/s eta 0:00:28
     ---------------------------------------- 0.7/198.6 MB 8.5 MB/s eta 0:00:24
     ---------------------------------------- 1.0/198.6 MB 7.6 MB/s eta 0:00:26
     ---------------------------------------- 1.0/198.6 MB 7.4 MB/s eta 0:00:27
     ---------------------------------------- 1.1/198.6 MB 4.2 


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import os
from facenet_pytorch import MTCNN, InceptionResnetV1
from PIL import Image
import cv2
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import pickle

print("Imports complete.")

c:\Users\frent\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports complete.


In [3]:
# 2. Configuration
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Running on device: {device}')

# Path to your dataset
data_dir = 'training_images_augmented' 

# Initialize MTCNN for face detection
# keep_all=False because we want the single best face for training embeddings.
mtcnn = MTCNN(
    image_size=160, margin=0, min_face_size=20,
    thresholds=[0.6, 0.7, 0.7], factor=0.709, post_process=True,
    device=device
)

# Initialize Inception Resnet V1 for face recognition
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

print("Models loaded.")

Running on device: cpu


100%|██████████| 107M/107M [00:19<00:00, 5.85MB/s] 


Models loaded.


In [4]:
def load_and_embed_images(data_dir):
    """
    Iterates through the dataset, detects faces, and generates embeddings.
    Returns:
        embeddings (list: numpy array): List of embeddings.
        labels (list: str): corresponding labels (names).
        name_map (dict): mapping from label index to name.
    """
    embeddings = []
    labels = []
    
    # Get all class names (subdirectories)
    class_names = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]
    print(f"Classes found: {class_names}")

    for idx, name in enumerate(class_names):
        class_dir = os.path.join(data_dir, name)
        print(f"Processing class: {name}")
        image_files = [f for f in os.listdir(class_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

        for file_name in image_files:
            img_path = os.path.join(class_dir, file_name)
            try:
                img = Image.open(img_path)
                
                # Check if image is RGB, if not convert
                if img.mode != 'RGB':
                    img = img.convert('RGB')

                # Get cropped face tensor directly from MTCNN
                # return_prob=False means it returns only the face tensor
                face_tensor = mtcnn(img) 

                if face_tensor is not None:
                    # Pass through Inception Resnet to get embedding
                    # Add batch dimension: [1, 3, 160, 160]
                    face_tensor = face_tensor.unsqueeze(0).to(device) 
                    
                    # Calculate embedding (detach from graph for numpy conversion)
                    emb = resnet(face_tensor).detach().cpu().numpy()
                    
                    embeddings.append(emb.flatten())
                    labels.append(name)
                else:
                    # Optional: Log if no face detected in an image
                    pass 

            except Exception as e:
                print(f"Error processing {img_path}: {e}")
                continue

    return np.array(embeddings), np.array(labels)

# Run the processing
print("Starting data processing (this may take a while)...")
embeddings, labels = load_and_embed_images(data_dir)
print(f"Start processing complete.")
print(f"Embeddings shape: {embeddings.shape}")
print(f"Labels shape: {labels.shape}")

Starting data processing (this may take a while)...
Classes found: ['Abhiram', 'Frentzen', 'Jessica', 'Ninad', 'Ryan', 'Sasi', 'Shreyas']
Processing class: Abhiram
Processing class: Frentzen
Processing class: Jessica
Processing class: Ninad
Processing class: Ryan
Processing class: Sasi
Processing class: Shreyas
Start processing complete.
Embeddings shape: (4135, 512)
Labels shape: (4135,)


In [5]:
# Encode labels
le = LabelEncoder()
labels_enc = le.fit_transform(labels)

print(f"Classes: {le.classes_}")

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(embeddings, labels_enc, test_size=0.2, random_state=42)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")

Classes: ['Abhiram' 'Frentzen' 'Jessica' 'Ninad' 'Ryan' 'Sasi' 'Shreyas']
Training samples: 3308
Testing samples: 827


In [6]:
# Train SVM Classifier
model = SVC(kernel='linear', probability=True)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy * 100:.2f}%")

from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred, target_names=le.classes_))

Accuracy: 99.52%
              precision    recall  f1-score   support

     Abhiram       0.98      0.99      0.98       132
    Frentzen       1.00      1.00      1.00       115
     Jessica       1.00      1.00      1.00       121
       Ninad       1.00      1.00      1.00       113
        Ryan       1.00      1.00      1.00       122
        Sasi       0.99      1.00      1.00       104
     Shreyas       1.00      0.97      0.99       120

    accuracy                           1.00       827
   macro avg       1.00      1.00      1.00       827
weighted avg       1.00      1.00      1.00       827



In [7]:
# Save Model and Label Encoder
with open('svm_classifier.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

print("Model and Label Encoder saved successfully.")

Model and Label Encoder saved successfully.


In [8]:
# Live Recognition Function
# Use this to verify the system works in real-time

def live_recognition():
    # Initialize separate MTCNN for detection (keep_all=True to detect all faces)
    mtcnn_detect = MTCNN(keep_all=True, device=device)
    
    cap = cv2.VideoCapture(0)
    print("Starting webcam... Press 'q' to quit.")

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        # Convert to RGB (OpenCV uses BGR)
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(frame_rgb)
        
        # Detect faces
        boxes, _ = mtcnn_detect.detect(pil_img)
        
        if boxes is not None:
            # If faces found, get aligned crops
            # MTCNN() call on image returns the *processed* tensors directly
            # but we need to know WHICH box corresponds to WHICH tensor.
            # mtcnn_detect(pil_img) returns (N, 3, 160, 160)
            
            # Efficient way: Get tensors and boxes together?
            # The library doesn't easily return both in one pass.
            # We will use the detected boxes to crop and feed to ResNet.
            
            for box in boxes:
                # Draw Box
                x1, y1, x2, y2 = [int(b) for b in box]
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                
                # Extract and Preprocess Face
                # Ensure coordinates are within image bounds
                h, w, _ = frame.shape
                x1 = max(0, x1); y1 = max(0, y1); x2 = min(w, x2); y2 = min(h, y2)
                
                face_img = pil_img.crop((x1, y1, x2, y2))
                face_img = face_img.resize((160, 160))
                
                # Convert to tensor and normalize (standard for InceptionResnetV1 in this lib)
                face_tensor = transforms.ToTensor()(face_img)
                # Standardize (approximate fixed_image_standardization)
                # (x - 127.5) / 128.0 is standard, but ToTensor gives [0,1].
                # So (x - 0.5) / 0.5 roughly scales to [-1, 1].
                # Actually facenet-pytorch uses: (x - 127.5) / 128.0 if input is uint8 [0,255].
                # Since ToTensor gives [0, 1], we do (x - 0.5) / 0.5019?
                # The library provides `fixed_image_standardization`
                
                # Let's use the explicit whitening the model expects
                # (Assuming pretrained='vggface2', it expects normalized inputs)
                face_tensor = (face_tensor - 0.5) / 0.5 
                
                face_tensor = face_tensor.unsqueeze(0).to(device)
                
                # Recognition
                embed = resnet(face_tensor).detach().cpu().numpy()
                prediction = model.predict(embed)
                prob = model.predict_proba(embed)
                
                name = le.inverse_transform(prediction)[0]
                confidence = prob[0][prediction[0]]
                
                label_text = f"{name} ({confidence*100:.1f}%)"
                cv2.putText(frame, label_text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)

        cv2.imshow('Face Recognition', frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

# Run the live recognition
# live_recognition() 
# Uncomment to run